# Temel NLP Görevleri

Table of Content:

- [1. Metin Sınıflandırma (Text Classification)](#1-metin-sınıflandırma-text-classification)
- [2. Varlık İsmi Tanıma (Named Entity Recognition)](#2-varlık-i̇smi-tanıma-named-entity-recognition)
- [3. Morfolojik Analiz](#3-morfolojik-analiz)
- [4. Metin Parçası Etiketleme (Part of Speech)](#4-metin-parçası-etiketleme-part-of-speech)
- [5. Kelime Anlamı Belirsizliği Giderme (Word Sense Disambiguation)](#5-kelime-anlamı-belirsizliği-giderme-word-sense-disambiguation)
    - [5.a. NLTK](#5a-nltk)
    - [5.b. PYWSD](#5b-pywsd)
- [6. Duygu Analizi (Sentiment Analysis)](#6-duygu-analizi-sentiment-analysis)

## 1. Metin Sınıflandırma (Text Classification):


In [3]:
import pandas as pd

spam_df = pd.read_csv("SMSSpamCollection.csv", names=["class", "sms"])

In [37]:
import re
from bs4 import BeautifulSoup as bs
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# import nltk
# nltk.download("stopwords")

lemmatizer = WordNetLemmatizer()

stop_words_eng = stopwords.words("english")

def clean_text(text):
    text = text.lower()
    text = bs(text, "html.parser").get_text()

    # Text temizliği:
    # Kelimeler arasındaki '-' karakterleri
    # Lookarounds:   (?=) - positive lookahead
    #                (?!) - negative lookahead
    #                (?<=) - positive lookbehind
    #                (?<!) - negative lookbehind
    text = re.sub(r"(?<=\w)-(?=\w)", " ", text)
    text = re.sub(r"(?<=\w)'(?=\w)", " ", text)
    text = re.sub(r"ll", " will", text)
    # Harf olmayan karakterlerin tamamı:
    text = re.sub(r"[^A-Za-z\s]", "", text)
    # Peşpeşe birden fazla kez gelen boşluk karakterleri:
    text = re.sub(r"\s{2,}", "", text)
    # Sık kullanılan farklı yazımlar: \b => word boundary 
    text = re.sub(r"cannot", "can not", text)
    text = re.sub(r"\bim\b", "i am", text)

    ttokens = text.split()
    
    text = " ".join([lemmatizer.lemmatize(word) for word in ttokens if word not in stop_words_eng])

    return text

In [17]:
X = spam_df["sms"].apply(clean_text)
y = spam_df["class"]

In [18]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state=60)

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
Xtr_BoW = cv.fit_transform(x_train)

In [22]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(Xtr_BoW, y_train)

Xte_BoW = cv.transform(x_test)

In [25]:
from sklearn.metrics import confusion_matrix

In [31]:
pred = dt.predict(Xte_BoW)
c_matrix = confusion_matrix(y_test, pred)
tn = c_matrix[0][0]
fn = c_matrix[0][1]
fp = c_matrix[1][0]
tp = c_matrix[1][1]
print(f"{c_matrix}\nPozitive Error Rate: %{(fn/(tn+fn))*100}\nNegative Error Rate: %{(fp/(tp+fp))*100}\nAccuracy: %{100-((fp+fn)/(fp+fn+tn+tp))*100}")

[[1553   22]
 [  63  202]]
Pozitive Error Rate: %1.3968253968253967
Negative Error Rate: %23.77358490566038
Accuracy: %95.3804347826087


## 2. Varlık İsmi Tanıma (Named Entity Recognition):

In [1]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.3 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
content = "I work at McDonald's and live in Wyoming. I would like to move to Los Angeles and work at Burger King this year."
doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.label_)

entities = [(ent.text, ent.text, ent.label_, ent.lemma_) for ent in doc.ents]

McDonald's ORG
Wyoming GPE
Los Angeles GPE
Burger King ORG
this year DATE


## 3. Morfolojik Analiz:

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [5]:
word = "Plural form of fish is also fish."

doc = nlp(word)

for token in doc:
    print(f"Text: {token.text}")
    print(f"Lemma: {token.lemma_}")
    print(f"POS: {token.pos_}")
    print(f"Tag: {token.tag_}")
    print(f"Is alpha: {token.is_alpha}")
    print(f"Is stop: {token.is_stop}")
    print(f"Morphology: {token.morph}\n")

Text: Plural
Lemma: plural
POS: ADJ
Tag: JJ
Is alpha: True
Is stop: False
Morphology: Degree=Pos

Text: form
Lemma: form
POS: NOUN
Tag: NN
Is alpha: True
Is stop: False
Morphology: Number=Sing

Text: of
Lemma: of
POS: ADP
Tag: IN
Is alpha: True
Is stop: True
Morphology: 

Text: fish
Lemma: fish
POS: NOUN
Tag: NN
Is alpha: True
Is stop: False
Morphology: Number=Sing

Text: is
Lemma: be
POS: AUX
Tag: VBZ
Is alpha: True
Is stop: True
Morphology: Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin

Text: also
Lemma: also
POS: ADV
Tag: RB
Is alpha: True
Is stop: True
Morphology: 

Text: fish
Lemma: fish
POS: NOUN
Tag: NN
Is alpha: True
Is stop: False
Morphology: Number=Sing

Text: .
Lemma: .
POS: PUNCT
Tag: .
Is alpha: False
Is stop: False
Morphology: PunctType=Peri



## 4. Metin Parçası Etiketleme (Part of Speech):

In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [3]:
sentence = "I don't like summer because it is too hot. I prefer winter."

doc = nlp(sentence)
for token in doc:
    print(token.text, token.pos_)

I PRON
do AUX
n't PART
like VERB
summer NOUN
because SCONJ
it PRON
is AUX
too ADV
hot ADJ
. PUNCT
I PRON
prefer VERB
winter NOUN
. PUNCT


## 5. Kelime Anlamı Belirsizliği Giderme (Word Sense Disambiguation):

### 5.a. NLTK:

In [6]:
import nltk
from nltk.wsd import lesk

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
s1 = "I went to the bank to deposit my money."
s2 = "The river bank was full of fish."

meaning1 = lesk(nltk.word_tokenize(s1), "bank")
meaning2 = lesk(nltk.word_tokenize(s2), "bank")

print(f"Sentence 1: {s1}\nMeaning of 'bank': {meaning1.definition()}")
print(f"Sentence 2: {s2}\nMeaning of 'bank': {meaning2.definition()}")

Sentence 1: I went to the bank to deposit my money.
Meaning of 'bank': a financial institution that accepts deposits and channels the money into lending activities
Sentence 2: The river bank was full of fish.
Meaning of 'bank': sloping land (especially the slope beside a body of water)


### 5.b. PYWSD:

In [9]:
%pip install pywsd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 1.4 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 1.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from pywsd.lesk import simple_lesk, adapted_lesk, cosine_lesk

In [12]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [20]:
sentences = [s1, s2]

for sentence in sentences:
    meaning_simple = simple_lesk(sentence, "bank")
    meaning_adapted = adapted_lesk(sentence, "bank")
    meaning_cosine = cosine_lesk(sentence, "bank")

    print(f"Sentence: {sentence}")
    print(f"Simple Lesk Meaning: {meaning_simple.definition()}")
    print(f"Adapted Lesk Meaning: {meaning_adapted.definition()}")
    print(f"Cosine Lesk Meaning: {meaning_cosine.definition()}\n")

Sentence: I went to the bank to deposit my money.
Simple Lesk Meaning: a financial institution that accepts deposits and channels the money into lending activities
Adapted Lesk Meaning: a financial institution that accepts deposits and channels the money into lending activities
Cosine Lesk Meaning: a container (usually with a slot in the top) for keeping money at home

Sentence: The river bank was full of fish.
Simple Lesk Meaning: sloping land (especially the slope beside a body of water)
Adapted Lesk Meaning: sloping land (especially the slope beside a body of water)
Cosine Lesk Meaning: a financial institution that accepts deposits and channels the money into lending activities



## 6. Duygu Analizi (Sentiment Analysis):

In [ ]:
import pandas as pd
import nltk

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize

In [23]:
imdb_df = pd.read_csv("IMDB Dataset.csv")
imdb_df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [38]:
X = imdb_df["review"].apply(clean_text)

In [ ]:
nltk.download("vader_lexicon")
nltk.download("punkt")
nltk.download("omw-1.4")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [49]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiments(text):
    scores = analyzer.polarity_scores(text)
    if scores["compound"] >= 0:
        return "positive"
    else:
        return "negative"

In [48]:
for i in range(5):
    [review, r_sentiment] = imdb_df.iloc[i]
    p_sentiment = get_sentiments(review)
    print(f"Review: {review[:50]}...\nSentiment: {p_sentiment}\tGuessed Sentiment: {r_sentiment}\n\n")

Review: One of the other reviewers has mentioned that afte...
Sentiment: negative	Guessed Sentiment: positive


Review: A wonderful little production. <br /><br />The fil...
Sentiment: positive	Guessed Sentiment: positive


Review: I thought this was a wonderful way to spend time o...
Sentiment: positive	Guessed Sentiment: positive


Review: Basically there's a family where a little boy (Jak...
Sentiment: negative	Guessed Sentiment: negative


Review: Petter Mattei's "Love in the Time of Money" is a v...
Sentiment: positive	Guessed Sentiment: positive




In [52]:
from sklearn.metrics import classification_report, confusion_matrix
p_sentiments = []
for i in range(1000):
    p_sentiments.append(get_sentiments(X[i]))
cm = confusion_matrix(imdb_df["sentiment"].iloc[:1000], p_sentiments)
cr = classification_report(imdb_df["sentiment"].iloc[:1000], p_sentiments)

print(f"Confusion matrix:\n{cm}\n")
print(f"Classification report:\n{cr}")

Confusion matrix:
[[245 254]
 [ 83 418]]

Classification report:
              precision    recall  f1-score   support

    negative       0.75      0.49      0.59       499
    positive       0.62      0.83      0.71       501

    accuracy                           0.66      1000
   macro avg       0.68      0.66      0.65      1000
weighted avg       0.68      0.66      0.65      1000

